In [1]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas kagglehub

import os
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw5_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text('SELECT version()')).scalar_one())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 28.1 MB/s eta 0:00:00
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [2]:
!pip install -q jupysql

%reload_ext sql
%sql engine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 995.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.5 MB/s eta 0:00:00


In [3]:
# Варіант A: якщо Kaggle credentials налаштовані
import kagglehub

dataset_dir = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(dataset_dir)

/kaggle/input/brazilian-ecommerce


In [4]:
files = {
    'olist_customers_raw': 'olist_customers_dataset.csv',
    'olist_orders_raw': 'olist_orders_dataset.csv',
    'olist_order_items_raw': 'olist_order_items_dataset.csv',
    'olist_products_raw': 'olist_products_dataset.csv',
    'olist_sellers_raw': 'olist_sellers_dataset.csv',
}

for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists='replace', index=False, chunksize=10_000)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)
olist_orders_raw (99441, 8)
olist_order_items_raw (112650, 7)
olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


In [5]:
%%sql

DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders      CASCADE;
DROP TABLE IF EXISTS olist_customers   CASCADE;
DROP TABLE IF EXISTS olist_products    CASCADE;
DROP TABLE IF EXISTS olist_sellers     CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);


Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

++
||
++
++

In [6]:
%%sql

INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

99441 rows affected.

99441 rows affected.

32951 rows affected.

3095 rows affected.

112650 rows affected.

++
||
++
++

In [7]:
%%sql

SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

5 rows affected.

table_name,n_rows
customers,99441
order_items,112650
orders,99441
products,32951
sellers,3095


## Завдання 2. CTE-chain pipeline з ≥5 stages

Feature engineering pipeline per customer: RFM (`recency_days`, `frequency`, `monetary`), `cohort_quarter`, `avg_order_value_30d` (rolling 30-day AVG), `cancellation_rate`, `preferred_category` (top-1 за category revenue).

`reference_date` обчислюється з датасету (`MAX(order_purchase_timestamp)::date + 1`), а не через `CURRENT_DATE`, щоб результат був відтворюваним.

In [8]:
%%sql

WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 20;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

20 rows affected.

customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21
3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0000,artes,6726.66
05455dfa7cd02f13d132aa7a6a9729c6,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0000,agro_industria_e_comercio,6081.54
df55c14d1476a9a3467f131269c2477f,RJ,565,1,4950.34,2017-04-01,2017-04-01,4950.34,0.0000,eletroportateis,4950.34
e0a2412720e9ea4f26c1ac985f6a7358,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0000,cool_stuff,4809.44
24bbf5fd2f2e1b359ee7de94defc4a15,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0000,eletroportateis,4764.34
3d979689f636322c62418b6346b1c6d2,PB,118,1,4681.78,2018-06-22,2018-04-01,4681.78,0.0000,pcs,4681.78


# Завдання 3. Recursive CTE для referral chain

In [9]:
%%sql


DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT
    prev.customer_id AS referrer_customer_id,
    nxt.customer_id AS referred_customer_id
FROM ordered_customers AS prev
JOIN ordered_customers AS nxt
    ON nxt.rn = prev.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;

SELECT *
FROM hw5_referrals
ORDER BY referrer_customer_id, referred_customer_id;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

9 rows affected.

9 rows affected.

referrer_customer_id,referred_customer_id,referred_at
00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2026-10-31 21:49:46.031131+00:00
000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079,2026-10-31 21:49:46.031131+00:00
0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5,2026-10-31 21:49:46.031131+00:00
0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb,2026-10-31 21:49:46.031131+00:00
000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652,2026-10-31 21:49:46.031131+00:00
0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086,2026-10-31 21:49:46.031131+00:00
000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282,2026-10-31 21:49:46.031131+00:00
00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c,2026-10-31 21:49:46.031131+00:00
00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130,2026-10-31 21:49:46.031131+00:00


In [10]:
%%sql

WITH RECURSIVE referral_depth AS (
    -- ANCHOR: root customers, які не були referred іншими customers
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: перейти від referrer до його referred customers
    SELECT
        r.referred_customer_id AS customer_id,
        rd.depth + 1 AS depth,
        rd.path_ids || r.referred_customer_id AS path_ids,
        rd.path_text || ' -> ' || r.referred_customer_id AS path_text
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
      AND rd.depth < 50
)
SELECT
    depth,
    COUNT(*) AS n_customers,
    MAX(path_text) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

10 rows affected.

depth,n_customers,sample_path
1,99432,ffffe8b65bbe3087b653a978c870db99
2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140
3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079
4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5
5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb
6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652
7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086
8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282
9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282 -> 00050bf6e01e69d5c0fd612f1bcfb69c
10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282 -> 00050bf6e01e69d5c0fd612f1bcfb69c -> 000598caf2ef4117407665ac33275130


### Reflection: що станеться при циклі A → B → A

**Без захистів.** Recursive CTE з `UNION ALL` не має вбудованої зупинки: recursive term виконується доти, доки повертає нові рядки. Якщо traversal дійде до A, далі піде B, потім знову A, потім B — і так нескінченно. Кожна ітерація дає “новий” рядок (інші `depth`, довші `path_ids`/`path_text`), тому навіть `UNION` замість `UNION ALL` не допоможе: рядки не є точними дублікатами. Запит працюватиме, доки не впаде по пам’яті/диску або по `statement_timeout`, а `path_text` ростиме з кожним кроком.


**Production-захисти (обидва вже є в запиті вище):**

1. **Path tracking.** Масив `path_ids` зберігає всіх клієнтів на поточному шляху, а умова `WHERE NOT r.referred_customer_id = ANY(rd.path_ids)` не дає перейти до вузла, який уже є на шляху. Для A → B → A гілка зупиниться на B, бо A вже в `path_ids`. Це логічний захист, який точно виявляє цикл.
2. **Depth guard.** `rd.depth < 50` — жорстка межа глибини. Він не виявляє цикл, а обмежує шкоду: навіть якщо path-перевірка буде зламана або граф виявиться аномально глибоким, recursion зупиниться максимум на 50 рівнях. Це safety net.

**Альтернатива: CYCLE-clause (PostgreSQL 14+).** Замість ручного масиву можна написати після CTE:

```sql
CYCLE customer_id SET is_cycle USING cycle_path
```

PostgreSQL сам веде шлях і позначає рядок `is_cycle = true`, коли вузол повторюється; далі такі рядки фільтруються `WHERE NOT is_cycle`. Depth guard при цьому все одно варто залишити.

# Завдання 4. Materialized View з CONCURRENT REFRESH

In [11]:
%%sql

DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

CREATE MATERIALIZED VIEW mv_olist_customer_features AS
WITH
reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id;

CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features (customer_id);


Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

99441 rows affected.

++
||
++
++

In [12]:
%%sql

SELECT COUNT(*) AS n_rows
FROM mv_olist_customer_features;



Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

1 rows affected.

n_rows
99441


In [13]:
%%sql

SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

5 rows affected.

customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21
3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0000,artes,6726.66


In [14]:
%%sql

REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;


Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

++
||
++
++

# Завдання 5. EXPLAIN ANALYZE + Reflection

In [16]:
%config SqlMagic.displaylimit = 1000

In [17]:
%%sql


EXPLAIN (ANALYZE, BUFFERS)
WITH
reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
)
SELECT *
FROM stage2_rfm
ORDER BY monetary DESC NULLS LAST
LIMIT 10;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

32 rows affected.

QUERY PLAN
Limit (cost=33611.33..33611.36 rows=10 width=65) (actual time=584.223..584.233 rows=10 loops=1)
"Buffers: shared hit=105261, temp read=961 written=1696"
-> Sort (cost=33611.33..33611.83 rows=200 width=65) (actual time=584.221..584.229 rows=10 loops=1)
Sort Key: stage2_rfm.monetary DESC NULLS LAST
Sort Method: top-N heapsort Memory: 27kB
"Buffers: shared hit=105261, temp read=961 written=1696"
-> Subquery Scan on stage2_rfm (cost=33601.51..33607.01 rows=200 width=65) (actual time=434.990..558.573 rows=99441 loops=1)
"Buffers: shared hit=105261, temp read=961 written=1696"
-> HashAggregate (cost=33601.51..33605.01 rows=200 width=69) (actual time=434.988..548.819 rows=99441 loops=1)
"Group Key: o.customer_id, (((max(olist_orders.order_purchase_timestamp))::date + 1))"


In [18]:
%%sql

CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC);

EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

17 rows affected.

QUERY PLAN
Limit (cost=4449.01..4450.16 rows=10 width=92) (actual time=25.965..27.337 rows=10 loops=1)
Buffers: shared hit=1636
-> Gather Merge (cost=4449.01..11175.94 rows=58495 width=92) (actual time=25.963..27.333 rows=10 loops=1)
Workers Planned: 1
Workers Launched: 1
Buffers: shared hit=1636
-> Sort (cost=3449.00..3595.24 rows=58495 width=92) (actual time=23.195..23.197 rows=7 loops=2)
Sort Key: monetary DESC NULLS LAST
Sort Method: top-N heapsort Memory: 26kB
Buffers: shared hit=1636


## Reflection


In [19]:
%%sql

-- Feature: avg_order_value_30d (stage1 + stage4, window functions)
EXPLAIN (ANALYZE, BUFFERS)
WITH
stage1_orders_total AS (
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY o.customer_id, o.order_id, o.order_status, o.order_purchase_timestamp
),
stage4_rolling_aov AS (
    SELECT
        s.customer_id,
        s.order_id,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
)
SELECT *
FROM stage4_rolling_aov
WHERE rn_recent = 1
ORDER BY avg_order_value_30d DESC
LIMIT 10;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

40 rows affected.

QUERY PLAN
Limit (cost=52350.55..52350.57 rows=10 width=90) (actual time=891.554..891.562 rows=10 loops=1)
"Buffers: shared hit=103453, temp read=1181 written=1184"
-> Sort (cost=52350.55..52351.79 rows=497 width=90) (actual time=891.552..891.558 rows=10 loops=1)
Sort Key: stage4_rolling_aov.avg_order_value_30d DESC
Sort Method: top-N heapsort Memory: 26kB
"Buffers: shared hit=103453, temp read=1181 written=1184"
-> Subquery Scan on stage4_rolling_aov (cost=41203.19..52339.81 rows=497 width=90) (actual time=555.089..864.881 rows=99441 loops=1)
Filter: (stage4_rolling_aov.rn_recent = 1)
"Buffers: shared hit=103453, temp read=1181 written=1184"
-> WindowAgg (cost=41203.19..51096.80 rows=99441 width=98) (actual time=555.086..853.502 rows=99441 loops=1)


In [20]:
%%sql

-- Feature: preferred_category (stage6 + stage7, 3-table JOIN + ROW_NUMBER)
EXPLAIN (ANALYZE, BUFFERS)
WITH
stage6_category_revenue AS (
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    SELECT customer_id, product_category_name, category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT *
FROM stage7_preferred_category
ORDER BY category_revenue DESC
LIMIT 10;

Running query in 'postgresql://postgres:***@/postgres?host=%2Ftmp%2Fhw5_pg'

54 rows affected.

QUERY PLAN
Limit (cost=38561.63..38561.65 rows=10 width=81) (actual time=1507.451..1510.937 rows=10 loops=1)
"Buffers: shared hit=5032, temp read=996 written=998"
-> Sort (cost=38561.63..38563.03 rows=563 width=81) (actual time=1507.449..1510.933 rows=10 loops=1)
Sort Key: ranked.category_revenue DESC
Sort Method: top-N heapsort Memory: 26kB
"Buffers: shared hit=5032, temp read=996 written=998"
-> Subquery Scan on ranked (cost=16948.65..38549.46 rows=563 width=81) (actual time=788.128..1460.891 rows=98666 loops=1)
Filter: (ranked.rn = 1)
"Buffers: shared hit=5032, temp read=996 written=998"
-> WindowAgg (cost=16948.65..37141.34 rows=112650 width=89) (actual time=788.126..1441.028 rows=98666 loops=1)


**1. Найдорожчий feature.** `preferred_category`: 1512 ms проти 893 ms для `avg_order_value_30d` і 586 ms для RFM. Сам JOIN дешевий: `Parallel Hash Join` orders↔items і `Hash Left Join` з products (усі таблиці через `Seq Scan`) завершуються за ~330 ms. Решта часу — `Sort` за `(customer_id, category)` з `external merge` (4992 kB + 2976 kB на диску), `GroupAggregate` до ~1244 ms, потім `Incremental Sort` і `WindowAgg` для `ROW_NUMBER`. Bottleneck — сортування, яке не вміщується у `work_mem`. Та сама проблема в інших features: у rolling AOV `Sort` перед `WindowAgg` — `external merge Disk: 9448kB`, у RFM `HashAggregate` має `Batches: 21, Disk Usage: 7400kB`, бо планувальник очікував 200 груп замість 99 441.

**2. Кешування.** Так. Читання з `mv_olist_customer_features` займає 27 ms (`Parallel Seq Scan` + `top-N heapsort`) — у ~55 разів швидше за перерахунок лише preferred_category. Refresh — щодоби через `REFRESH ... CONCURRENTLY` (унікальний індекс по `customer_id` уже є): features мають денну гранулярність (`recency_days`, 30-денне вікно), тож щогодинний refresh нічого не додає. Для статичного Olist достатньо on-demand після завантаження даних. Індекс `(monetary DESC)` не використався, бо запит сортує `NULLS LAST`, а індекс — `DESC NULLS FIRST`; потрібен `(monetary DESC NULLS LAST)`.

**3. Коли VIEW кращий.** Коли потрібен актуальний стан для одного клієнта: support-оператор дивиться `cancellation_rate` одразу після скасування замовлення. MV показала б стан на момент refresh, а перерахунок для одного `customer_id` дешевий (з індексом на `olist_orders(customer_id)`).

**4. Termination.** Recursive term виконується, поки повертає нові рядки; на листках referral chain working table стає порожньою. Цикл `A → B → A` без захистів дає нескінченну рекурсію; `NOT ... = ANY(path_ids)` обриває гілку, а `depth < 50` — страхувальна межа (детальніше — у Завданні 3).

**5. Trade-off.** Recompute-cost: один feature коштує 1,5 s зі спілом на диск, а дані змінюються рідко, тож freshness можна пожертвувати; storage для 99 441 рядка незначний.